# Classificador 4 – Lesões de Baixo Grau (LSIL vs ASC-US)

In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from tqdm import tqdm

In [ ]:
IMG_SIZE = 70
BATCH_SIZE = 32
EPOCHS = 30
NUM_CLASSES = 2
KFOLD_SPLITS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "../data/crops/c4_baixo_grau"
OUTPUT_DIR = "../results/c4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

In [14]:
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
print("Classes:", dataset.classes)
print("Total de imagens:", len(dataset))

Classes: ['ASCUS', 'LSIL']
Total de imagens: 2710


In [15]:
kf = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
fold_metrics = []
for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"===== Fold {fold+1} =====")
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0003)
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Fold {fold+1} - Época {epoch+1}"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1} - Loss médio: {running_loss/len(train_loader):.4f}")
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.numpy())
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=dataset.classes, yticklabels=dataset.classes, cmap="Blues")
    plt.title(f"Matriz de Confusão - Fold {fold+1}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confusao_fold{fold+1}.png")
    plt.close()
    fold_metrics.append({
        "fold": fold + 1,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1
    })
    print(f"Acurácia : {acc:.4f}")
    print(f"Precisão : {prec:.4f}")
    print(f"Revocação: {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")

===== Fold 1 =====


Fold 1 - Época 1: 100%|██████████| 57/57 [00:07<00:00,  8.14it/s]


Epoch 1 - Loss médio: 0.5631


Fold 1 - Época 2: 100%|██████████| 57/57 [00:02<00:00, 20.23it/s]


Epoch 2 - Loss médio: 0.4424


...

Epoch 8 - Loss médio: 0.1936


Fold 3 - Época 9: 100%|██████████| 57/57 [00:02<00:00, 21.71it/s]


Epoch 9 - Loss médio: 0.1602


Fold 3 - Época 10: 100%|██████████| 57/57 [00:02<00:00, 21.39it/s]


Epoch 10 - Loss médio: 0.1599
Acurácia : 0.8195
Precisão : 0.8195
Revocação: 0.8196
F1-Score : 0.8195


In [16]:
df_metrics = pd.DataFrame(fold_metrics)
df_metrics.loc["média"] = df_metrics.mean(numeric_only=True)
df_metrics.to_csv(f"{OUTPUT_DIR}/metricas_classificador4.csv", index=False)
df_metrics

,fold,accuracy,precision,recall,f1_score
0,1.0,0.830752,0.831721,0.831485,0.830747
1,2.0,0.827243,0.826684,0.826973,0.826812
2,3.0,0.819491,0.819522,0.819606,0.819483
média,2.0,0.825828,0.825976,0.826022,0.825681
